# Hotel Reservation Chatbot (v3)

- this version will remember everything
- this version will have an access to the hotel database

### install required packages

In [ ]:
%pip install langchain langchain_community langchain_openai langgraph

### import required packages

In [1]:
import sqlite3
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

### create the database

In [2]:
#connect to SQLite DB
conn = sqlite3.connect('hotel.db')
cursor = conn.cursor()

# Create the rooms table
cursor.execute('''
CREATE TABLE IF NOT EXISTS rooms (
  room_number INTEGER PRIMARY KEY,
  room_type TEXT NOT NULL,
  price REAL NOT NULL,
  max_capacity INTEGER NOT NULL,
  amenities TEXT
)
''')

# Create the reservations table
cursor.execute('''
CREATE TABLE IF NOT EXISTS reservations (
  reservation_id INTEGER PRIMARY KEY AUTOINCREMENT,
  guest_name TEXT NOT NULL,
  room_number INTEGER NOT NULL,
  start_date DATE NOT NULL,
  end_date DATE NOT NULL,
  FOREIGN KEY (room_number) REFERENCES rooms(room_number)
)
''')


# Example data for rooms
rooms_data = [
  (101, 'Single', 100, 1, 'TV, Wi-Fi'),
  (102, 'Double', 150, 2, 'TV, Wi-Fi, Balcony'),
  (103, 'Suite', 250, 3, 'TV, Wi-Fi, Balcony, Jacuzzi'),
  (104, 'Deluxe', 200, 2, 'TV, Wi-Fi, Mini-bar'),
  (105, 'Family Suite', 300, 4, 'TV, Wi-Fi, Balcony, Kitchenette'),
  (106, 'Executive Suite', 350, 2, 'TV, Wi-Fi, Balcony, Jacuzzi, City View'),
  (107, 'Double', 150, 2, 'TV, Wi-Fi'),
  (108, 'Deluxe', 200, 2, 'TV, Wi-Fi, Balcony, Fireplace')
]

# Insert the data
cursor.executemany('INSERT OR IGNORE INTO rooms VALUES (?, ?, ?, ?, ?)', rooms_data)

reservations_data = [
    ('Alice Smith', 102, '2025-03-15', '2025-03-18'),
    ('Bob Johnson', 105, '2025-04-22', '2025-04-25'),
    ('Charlie Brown', 106, '2025-06-10', '2025-06-12')
]

# Insert the data
cursor.executemany('INSERT INTO reservations (guest_name, room_number, start_date, end_date) VALUES (?, ?, ?, ?)', reservations_data)

conn.commit()
conn.close()
print("Database initialized successfully.")

Database initialized successfully.


### create llm

In [3]:
# load all keys from .env file
load_dotenv()

# create llm object
llm = ChatOpenAI(model="gpt-5", temperature=1.0)

### configure the prompts

In [7]:
# MessagesState contains a key called as "messages" which is of type a list
from langgraph.graph import MessagesState
from langchain_core.messages import SystemMessage

# System message for Conversational Agent
sys_msg_conversation = SystemMessage(content="""

  Role: You are the Conversation Coordinator for a hotel chain’s customer support system. Your role is to manage the flow of the conversation with the guest and decide whether to handle the request yourself or forward the conversation to the next agent based on the user’s needs.

  Primary Tasks:

  Engage with the user: Start the conversation politely and ask how you can assist them.
  Identify the nature of the request: Listen to the guest’s inquiry and determine if it’s a general question, booking-related, issue resolution, or any other category that might require specialized handling.
  Route the conversation if necessary: If the request requires expertise outside your scope, forward it to a specialized agent by answering with the name of the corresponding agent. There are the following agents available:

  - reservation_assistant;

  Maintain a friendly and helpful tone: Always aim for a customer-friendly and empathetic response. Ensure the guest feels heard and cared for.

  Guidelines:

  - For general inquiries (e.g., facilities, amenities, location details), assist the user directly.
  - If user wants to book a room. The following information: room type, required dates. Only if user has provided this information forward to the reservation_assistant otherwise do not mention reservation_assistant.
    
  """)


# System message for sql-based reservation agent
sys_msg_sql = SystemMessage(content="""

  You are AI assistant specialized in SQL. You have access to the following database schema:

  Table: rooms
    room_number (INTEGER PRIMARY KEY)
    room_type (TEXT NOT NULL)
    price (REAL NOT NULL)
    max_capacity (INTEGER NOT NULL)
    amenities (TEXT)

  Table: reservations
    reservation_id (INTEGER PRIMARY KEY AUTOINCREMENT)
    guest_name (TEXT NOT NULL)
    room_number (INTEGER NOT NULL, FOREIGN KEY REFERENCES rooms(room_number))
    start_date (DATE NOT NULL)
    end_date (DATE NOT NULL)

  You must:
    Provide SQL queries or instructions referencing only these two tables and the columns defined above.
    Generate valid SQL statements that accurately address the user’s questions or requests.
    Include explanations about how the queries work or how they address the request.
    Use best practices for SQL (proper joins, filters, etc.).
    Do not reference any tables or columns other than rooms and reservations.
    If the user’s request is unclear or not possible with the given schema, ask for clarification or note the limitations.
    If user want to book a room check if the room is not already booked for the corresponding dates

   """)

In [8]:
# Agent node
def conversation_assistant(state: MessagesState) -> MessagesState:
  """
  Conversation Assistant Agent

  Args:
    state (MessagesState): The current state of the conversation.

  Returns:
    MessagesState: The updated state of the conversation.
  """
  messages = state["messages"]
  return {"messages": [llm.invoke([sys_msg_conversation] + messages)]}

### create a tool for database access

In [11]:
from langchain_community.utilities import SQLDatabase
from langchain_community.agent_toolkits import SQLDatabaseToolkit

# define our db as a set of sql tools
db = SQLDatabase.from_uri("sqlite:///hotel.db")
toolkit = SQLDatabaseToolkit(db=db, llm=llm)
sql_tools = toolkit.get_tools()

# bind LLM with tools
llm_with_sql_tools = llm.bind_tools(tools=sql_tools)

In [12]:
# SQL Agent node
def sql_assistant(state: MessagesState) -> MessagesState:
  """
   SQL Assistant Agent

   Args:
     state (MessagesState): The current state of the conversation.

   Returns:
     MessagesState: The updated state of the conversation.
   """
  messages = state["messages"]
  return {"messages": [llm_with_sql_tools.invoke([sys_msg_sql] + messages)]}

### create a graph and configure it

In [14]:
from typing import Literal

# used for conditional node
def choose_next_node(state: MessagesState) -> Literal["reservation_assistant", "__end__"]:
  """
  Choose the next node based on the last message.

  Args:
    state (MessagesState): The current state of the conversation.

  Returns:
    str: The name of the next node.
  """
  last_message = state["messages"][-1]
  if ("reservation_assistant" in last_message.content) or ("reservation assistant" in last_message.content):
    return "reservation_assistant"
  else:
    return "__end__"

In [16]:
from langgraph.graph import START, END, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.memory import MemorySaver

# initialize the memory saver
memory = MemorySaver()

# Graph builder
builder = StateGraph(MessagesState)

# add a node to start conversation
builder.add_node("conversation_assistant", conversation_assistant)
builder.add_node("reservation_assistant", sql_assistant)
builder.add_node("sql_tools", ToolNode(sql_tools))

# add the required edges
builder.add_edge(START, "conversation_assistant")
builder.add_conditional_edges(
    "conversation_assistant",
    # If the message from assistant is a tool call -> routes to tools
    # Otherwise to the END
    choose_next_node, path_map=["reservation_assistant", "__end__"])
builder.add_conditional_edges("reservation_assistant", tools_condition, path_map={"tools": "sql_tools", "__end__": "__end__"})
builder.add_edge("sql_tools", "reservation_assistant")

# add the memory saver to the graph as a checkpointer
react_graph_with_memory = builder.compile(checkpointer=memory)

In [ ]:
# Draw the graph
from IPython.display import Image, display
png = react_graph_with_memory.get_graph().draw_mermaid_png()
display(Image(png))

### invoke the graph

In [23]:
from langchain_core.messages import HumanMessage

# thread id
thread_id = 100

while True:
    
    # user input
    input_query = input("your query: ")
    if input_query == "exit":
        break
    
    # call the graph with the input
    events = react_graph_with_memory.stream(
        {"messages": [HumanMessage(content=input_query)]}, 
        stream_mode="values",
        config={"thread_id": thread_id}
    )
    
    # iterate over the messages
    for event in events:
       event["messages"][-1].pretty_print()
       message_content= event["messages"][-1].content
       if "Answer:" in message_content:
          final_answer=message_content.split("Answer:",1)[1].strip()

your query:  hi


================================ Human Message =================================

hi
================================== Ai Message ==================================

Hello! Welcome to [Your Hotel Name]. How can I assist you today? Are you looking to ask about our amenities or location, need help with an existing stay, or would you like to book a room?


your query:  I want to book a room in your hotel in New York


================================ Human Message =================================

I want to book a room in your hotel in New York
================================== Ai Message ==================================

Wonderful—I'd be happy to help you book a room in New York. To get started, could you please share:

- Check-in and check-out dates
- Preferred room type (e.g., Standard, Deluxe, Suite; one king bed or two doubles)
- Number of guests (and ages of any children)
- Any neighborhood or specific property preference in NYC
- Any special requests or budget range

Once I have the dates and room type, I can proceed with availability and options.


your query:  I want to book a double room from '2025-03-15' to '2025-03-18'


...

your query:  okay. then lets go with suite room number 103


================================ Human Message =================================

okay. then lets go with suite room number 103
================================== Ai Message ==================================

reservation_assistant
================================== Ai Message ==================================
Tool Calls:
  sql_db_query_checker (call_3j1oepxhMPRyNJOnuqE13ByJ)
 Call ID: call_3j1oepxhMPRyNJOnuqE13ByJ
  Args:
    query: INSERT INTO reservations (guest_name, room_number, start_date, end_date)
SELECT 'amit', 103, '2025-04-22', '2025-04-25'
WHERE NOT EXISTS (
  SELECT 1 FROM reservations AS res
  WHERE res.room_number = 103
    AND res.start_date < '2025-04-25'
    AND res.end_date > '2025-04-22'
)
RETURNING reservation_id, guest_name, room_number, start_date, end_date;
================================= Tool Message =================================
Name: sql_db_query_checker

INSERT INTO reservations (guest_name, room_number, start_date, end_date)
SELECT 'amit', 103, '2025

your query:  oh.. I want to cancel my reservation with reservation id as 4


================================ Human Message =================================

oh.. I want to cancel my reservation with reservation id as 4
================================== Ai Message ==================================

reservation_assistant
================================== Ai Message ==================================
Tool Calls:
  sql_db_query_checker (call_0iBPx6NTCLNKpYGIodpOKzzT)
 Call ID: call_0iBPx6NTCLNKpYGIodpOKzzT
  Args:
    query: DELETE FROM reservations
WHERE reservation_id = 4
RETURNING reservation_id, guest_name, room_number, start_date, end_date;
================================= Tool Message =================================
Name: sql_db_query_checker

DELETE FROM reservations
WHERE reservation_id = 4
RETURNING reservation_id, guest_name, room_number, start_date, end_date;
================================== Ai Message ==================================
Tool Calls:
  sql_db_query (call_RifWcaBdGtyu9Om5Arw7GZMb)
 Call ID: call_RifWcaBdGtyu9Om5Arw7GZMb
  Args:
  

your query:  exit
